# Tugas Besar II IF3170 — Implementasi Spesifikasi Bonus

Notebook ini berisi implementasi khusus untuk spesifikasi **Bonus** Tugas Besar II IF3170 Inteligensi Artifisial (di luar bonus optimasi algoritma model / Bonus #6):

1. **Bonus #2 — Decision Tree Learning (1 poin)**: Visualisasi gambar percabangan pohon keputusan dengan parameter `top_n` untuk mengatur jumlah tingkat percabangan yang ditampilkan.
2. **Bonus #3 — Logistic Regression (2 poin)**: Video animasi garis kontur fungsi *log-loss* dan lintasan parameter $(\theta_0, \theta_1)$ selama proses *training*.
3. **Bonus #4 — Support Vector Machine (1 poin)**: Gambar bidang pemisah (*decision boundary*), pita *margin*, dan *support vectors* untuk setiap kernel (*Linear*, *Polynomial*, *Gaussian/RBF*) pada ruang 2D.
4. **Bonus #5 — Bayesian Network / Naive Bayes (1 poin)**: Gambar graf berarah yang menampilkan hubungan variabel kelas terhadap fitur serta dependensi empiris antarfitur.
5. **Bonus #7 — Cost-Sensitive Classification (2 poin)**: Penanganan klasifikasi berbiaya asimetris dengan asumsi *False Positive* (risiko *overbooking* pada Ronaldo Hotel™) $2\times$ lebih mahal daripada *False Negative* ($C_{FP} = 2, C_{FN} = 1$).


---
## 1. Environment Setup & Data Preparation

Tahap ini memuat pustaka, mengimpor kelas model *from scratch* dari `src/models/`, serta menyiapkan pembagian data latih dan validasi terstratifikasi (80:20, `SEED = 42`). Seluruh parameter *preprocessing* (imputasi, *standardization*, dan *one-hot encoding*) hanya di-*fit* pada data latih untuk mencegah *data leakage*.


In [ ]:
import os
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Pastikan folder src berada di sys.path agar modul models dapat diimpor
SRC_DIR = Path.cwd() if (Path.cwd() / "models").exists() else Path.cwd() / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from models.knn import KNN
from models.decision_tree import DecisionTree
from models.logistic_regression import LogisticRegression
from models.svm import SVC
from models.naive_bayes import NaiveBayes

# Konfigurasi global (konsisten dengan src/main.ipynb)
SEED = 42
DATA_DIR = SRC_DIR.parent / "data"
OUTPUT_DIR = SRC_DIR.parent / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)


seed_everything(SEED)

# Palet publikasi (figures4papers)
PALETTE = {
    "blue_main": "#0F4D92",
    "blue_secondary": "#3775BA",
    "green_2": "#AADCA9",
    "green_3": "#8BCF8B",
    "red_1": "#F6CFCB",
    "red_2": "#E9A6A1",
    "red_strong": "#B64342",
    "neutral": "#CFCECE",
    "teal": "#42949E",
    "violet": "#9A4D8E",
    "highlight": "#FFD700",
}

print(f"DATA_DIR   : {DATA_DIR}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")


In [ ]:
# Load dataset & Stratified Train-Validation Split (80:20)
train_df = pd.read_csv(DATA_DIR / "train.csv", dtype={"id": str})

train_set, val_set = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["is_canceled"],
    random_state=SEED,
)

# Pisahkan fitur mentah dan target
X_train_raw = train_set.drop(columns=["id", "is_canceled"]).copy()
y_train = train_set["is_canceled"].to_numpy(dtype=np.int64)

X_val_raw = val_set.drop(columns=["id", "is_canceled"]).copy()
y_val = val_set["is_canceled"].to_numpy(dtype=np.int64)

# Preprocessing pipeline (fit hanya pada train)
median_cols = [
    "num_adults",
    "num_children",
    "days_before_arrival",
    "nightly_rate_idr",
    "special_requests",
]
binary_cols = ["needs_parking", "is_returning_guest"]
categorical_cols = ["meal_package", "room_category", "booking_channel"]
numerical_cols = [
    "num_adults",
    "num_children",
    "weekend_nights",
    "weekday_nights",
    "days_before_arrival",
    "prior_cancellations",
    "prior_completed_bookings",
    "nightly_rate_idr",
    "special_requests",
    "arrival_year",
    "arrival_month",
    "arrival_day",
]

X_train = X_train_raw.copy()
X_val = X_val_raw.copy()

# 1. Imputasi
num_imp = SimpleImputer(strategy="median").fit(X_train[median_cols])
bin_imp = SimpleImputer(strategy="most_frequent").fit(X_train[binary_cols])
cat_imp = SimpleImputer(strategy="constant", fill_value="Unknown").fit(X_train[categorical_cols])

for frame in (X_train, X_val):
    frame[median_cols] = num_imp.transform(frame[median_cols])
    frame[binary_cols] = bin_imp.transform(frame[binary_cols])
    frame[categorical_cols] = cat_imp.transform(frame[categorical_cols])

# 2. Z-Score Standardization fitur numerik
scaler = StandardScaler().fit(X_train[numerical_cols])
for frame in (X_train, X_val):
    frame[numerical_cols] = scaler.transform(frame[numerical_cols])

# 3. One-Hot Encoding fitur kategorikal
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(X_train[categorical_cols])
ohe_names = list(ohe.get_feature_names_out(categorical_cols))

for frame in (X_train, X_val):
    encoded = ohe.transform(frame[categorical_cols])
    frame.drop(columns=categorical_cols, inplace=True)
    frame[ohe_names] = encoded

feature_names = list(X_train.columns)
X_train_final = X_train.to_numpy(dtype=np.float64)
X_val_final = X_val.to_numpy(dtype=np.float64)

print(f"X_train_final: {X_train_final.shape} | X_val_final: {X_val_final.shape}")
print(f"Distribusi target train -> 0: {(y_train == 0).mean():.2%}, 1: {(y_train == 1).mean():.2%}")


---
## 2. Bonus #2 — Visualisasi Percabangan Decision Tree (`top_n`)

Metode `DecisionTree.visualize_tree(filename, save_to_file=True, top_n=N, feature_names=...)` menghasilkan gambar percabangan pohon keputusan hingga kedalaman `top_n` tingkat teratas. Simpul yang dipotong pada kedalaman `top_n` diringkas menjadi label `... (pred: kelas_mayoritas)` sehingga struktur aturan paling dominan tetap mudah dibaca.


In [ ]:
# Latih DecisionTree from scratch (konfigurasi tuned CART)
dtl_bonus = DecisionTree(
    criterion="gini",
    use_gain_ratio=False,
    max_depth=8,
    min_samples_split=2,
    min_samples_leaf=1,
    ccp_alpha=0.0,
)
dtl_bonus.fit(X_train_final, y_train)

# Simpan visualisasi percabangan pohon dengan parameter top_n=3
dtl_top_n = 3
dtl_img_path = OUTPUT_DIR / "decision_tree_visualization.png"
dtl_bonus.visualize_tree(
    filename=str(dtl_img_path),
    save_to_file=True,
    top_n=dtl_top_n,
    feature_names=feature_names,
)

val_f1_dtl = f1_score(y_val, dtl_bonus.predict(X_val_final), average="macro")
print(f"DecisionTree (max_depth=8) Holdout Macro F1: {val_f1_dtl:.4f}")
print(f"Visualisasi top_n={dtl_top_n} disimpan ke: {dtl_img_path}")
display(Image(filename=str(dtl_img_path)))


---
## 3. Bonus #3 — Video Kontur *Log-Loss* & Lintasan Parameter Logistic Regression

Metode `LogisticRegression.animate_loss_contour(...)` merekam riwayat parameter selama pelatihan (`record_history=True`) lalu membuat video animasi (`.gif` / `.mp4`) yang menampilkan:
- **Panel Kiri**: Garis kontur permukaan fungsi *binary log-loss* terhadap $(\theta_0, \theta_1)$ beserta lintasan konvergensi dari inisialisasi awal $\theta^{(0)}$ menuju optimum $\theta^{(T)}$.
- **Panel Kanan**: Kurva penurunan *training log-loss* per iterasi.

Di sini kita memvisualisasikan parameter *intercept* $\theta_0$ (Bias) dan bobot fitur paling berpengaruh $\theta_1$ (`days_before_arrival`).


In [ ]:
# Ambil fitur utama: days_before_arrival (sudah terstandarisasi)
feat_idx_days = feature_names.index("days_before_arrival")
X_lr_1d = X_train_final[:, [feat_idx_days]]

lr_anim = LogisticRegression(
    learning_rate=0.1,
    num_iterations=120,
    batch_size=None,
    l1_lambda=0.0,
    l2_lambda=0.0,
    optimizer="sgd",
    random_state=SEED,
)
lr_anim.fit(X_lr_1d, y_train, record_history=True)

gif_path = OUTPUT_DIR / "logistic_regression_contour.gif"
snap_path = OUTPUT_DIR / "logistic_regression_contour_final.png"

lr_anim.animate_loss_contour(
    X_lr_1d,
    y_train,
    filename=str(gif_path),
    save_to_file=True,
    snapshot_filename=str(snap_path),
    param_mode="intercept_weight",
    feature_indices=(0,),
    grid_size=75,
    n_frames=60,
    fps=15,
    dpi=150,
    param_names=(r"$\theta_0$ (Bias)", r"$\theta_1$ (days_before_arrival)"),
)

print(f"Video animasi disimpan ke : {gif_path}")
print(f"Snapshot akhir disimpan ke: {snap_path}")
print(
    f"Loss awal: {lr_anim.history['loss'][0]:.4f} -> "
    f"Loss akhir (iter-{len(lr_anim.history['loss']) - 1}): {lr_anim.history['loss'][-1]:.4f}"
)
display(Image(filename=str(snap_path)))


---
## 4. Bonus #4 — Visualisasi Bidang Pemisah, *Margin*, & *Support Vectors* SVM (2D)

Kita mengambil 2 fitur numerik utama (`days_before_arrival` dan `nightly_rate_idr`) pada 200 sampel terstratifikasi agar batas keputusan $f(x) = 0$, pita *margin* $f(x) = \pm 1$, *margin support vectors* ($0 < \alpha_i < C$), dan *bounded support vectors* ($\alpha_i = C$) terlihat jelas untuk ketiga kernel wajib: **Linear**, **Polynomial ($d=3$)**, dan **Gaussian (RBF)**.


In [ ]:
# Pilih 2 fitur utama dan 200 sampel terstratifikasi untuk visualisasi 2D
feat_2d_names = ["days_before_arrival", "nightly_rate_idr"]
feat_2d_idx = [feature_names.index(c) for c in feat_2d_names]

X_svm_2d, _, y_svm_2d, _ = train_test_split(
    X_train_final[:, feat_2d_idx],
    y_train,
    train_size=200,
    stratify=y_train,
    random_state=SEED,
)

kernel_configs = [
    ("Linear Kernel", dict(C=1.0, kernel="linear", tol=1e-4, max_iter=20000)),
    ("Polynomial Kernel (d=3)", dict(C=1.0, kernel="poly", degree=3, gamma="scale", coef0=1.0, tol=1e-4, max_iter=20000)),
    ("Gaussian (RBF) Kernel", dict(C=1.0, kernel="rbf", gamma="scale", tol=1e-4, max_iter=20000)),
]

fig, axes = plt.subplots(1, 3, figsize=(18.5, 5.6))
fig.patch.set_facecolor("white")

fitted_svcs = []
for ax, (label, kwargs) in zip(axes, kernel_configs):
    svc_2d = SVC(**kwargs).fit(X_svm_2d, y_svm_2d)
    acc_2d = (svc_2d.predict(X_svm_2d) == y_svm_2d).mean()
    bin_svc = svc_2d.estimators[0][2]
    free_mask, _ = bin_svc._sv_masks()
    title = (
        f"{label}\n"
        f"Acc: {acc_2d:.1%} | SV: {len(bin_svc.support_vectors)} ({int(free_mask.sum())} margin)"
    )
    svc_2d.visualize_boundary(
        X_svm_2d,
        y_svm_2d,
        ax=ax,
        feature_names=feat_2d_names,
        class_names=("Not Canceled (0)", "Canceled (1)"),
        title=title,
        distinguish_sv=True,
        show_legend=False,
    )
    fitted_svcs.append(svc_2d)

# Legend gabungan tanpa frame di bawah ketiga panel
handles = fitted_svcs[0].estimators[0][2]._legend_handles(
    ("Not Canceled (0)", "Canceled (1)"),
    distinguish_sv=True,
)
fig.legend(
    handles=handles,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.04),
    ncol=4,
    frameon=False,
    fontsize=11,
)
fig.suptitle(
    "Perbandingan Bidang Pemisah, Margin, dan Support Vectors SVM (2D)",
    fontsize=15,
    fontweight="bold",
    y=1.02,
)
fig.tight_layout(rect=(0.0, 0.05, 1.0, 0.98), pad=1.2)

svm_img_path = OUTPUT_DIR / "svm_kernels_2d_boundary.png"
fig.savefig(svm_img_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.close(fig)

print(f"Visualisasi 3 kernel SVM disimpan ke: {svm_img_path}")
display(Image(filename=str(svm_img_path)))


---
## 5. Bonus #5 — Visualisasi Graf Bayesian Network / Naive Bayes

Metode `NaiveBayes.visualize_graph(...)` menggambarkan struktur graf berarah Naive Bayes ($C \rightarrow X_j$) pada 17 fitur asli (`X_train_raw`, sebelum *one-hot encoding*) dengan bobot sisi berupa *Mutual Information* $I(C; X_j)$ dalam satuan bit, dilengkapi busur merah putus-putus untuk 8 pasangan fitur dengan *Mutual Information* empiris antarfitur tertinggi yang diabaikan oleh asumsi independensi kondisional Naive Bayes.


In [ ]:
# Latih NaiveBayes from scratch pada fitur mentah (17 fitur asli)
nb_graph = NaiveBayes(alpha=0.5, n_bins=10, use_feature_weighting=False)
nb_graph.fit(X_train_raw, y_train)

nb_img_path = OUTPUT_DIR / "naive_bayes_graph.png"
nb_graph.visualize_graph(
    filename=str(nb_img_path),
    save_to_file=True,
    X=X_train_raw,
    target_name="is_canceled",
    show_feature_edges=True,
    max_feature_edges=8,
    min_feature_mi=0.02,
    formats=("png",),
    dpi=300,
)

val_f1_nb_raw = f1_score(y_val, nb_graph.predict(X_val_raw), average="macro")
print(f"NaiveBayes (raw features, n_bins=10, alpha=0.5) Holdout Macro F1: {val_f1_nb_raw:.4f}")
print(f"Visualisasi graf Naive Bayes disimpan ke: {nb_img_path}")
display(Image(filename=str(nb_img_path)))


---
## 6. Bonus #7 — *Cost-Sensitive Classification* ($C_{FP} = 2, C_{FN} = 1$)

Pada studi kasus **Ronaldo Hotel™**, kesalahan *False Positive* (memprediksi tamu akan membatalkan padahal tamu tetap datang sehingga kamar dijual ulang dan terjadi *overbooking*) diasumsikan **$2\times$ lebih mahal** dibandingkan *False Negative* (kamar kosong akibat pembatalan tak terduga).

Dengan matriks biaya $C_{FP} = 2$ dan $C_{FN} = 1$, berdasarkan *Bayes Optimal Decision Rule* (Elkan, 2001), prediksi positif ($\hat{y} = 1$, dibatalkan) optimal apabila:

$$C_{FP} \cdot (1 - p) \le C_{FN} \cdot p \iff 2(1 - p) \le p \iff p \ge \tau^* = \frac{C_{FP}}{C_{FP} + C_{FN}} = \frac{2}{3} \approx 0.667$$

### Tiga Perubahan Penanganan pada Task Ini
1. **Metrik Evaluasi**: Metrik bisnis utama bergeser dari sekadar *Macro F1* menjadi **Total Cost** ($2 \cdot \text{FP} + \text{FN}$) dan **Average Cost** per pemesanan.
2. **Thresholding Asimetris**: Ambang keputusan probabilitas dinaikkan dari $\tau = 0.5$ ke ambang teoritis *Bayes Optimal* ($\tau^* = 2/3$) serta pencarian *Empirical Cost-Optimal Threshold* (dan *margin threshold* $z \ge z^*$ untuk SVM).
3. **Hindari *Naive Oversampling***: Teknik seperti SMOTE justru meningkatkan *False Positive* sehingga memperbesar biaya *overbooking*.


In [ ]:
# Latih kelima model tuned from scratch pada data terstandarisasi (X_train_final)
C_FP = 2.0
C_FN = 1.0
TAU_BAYES = C_FP / (C_FP + C_FN)  # 2/3 ≈ 0.6667

prob_models = {
    "KNN": KNN(k=15, metric="manhattan", voting_strategy="distance"),
    "DecisionTree": DecisionTree(
        criterion="gini",
        use_gain_ratio=False,
        max_depth=8,
        min_samples_split=2,
        min_samples_leaf=1,
        ccp_alpha=0.0,
    ),
    "LogisticRegression": LogisticRegression(
        learning_rate=0.1,
        num_iterations=500,
        batch_size=256,
        l2_lambda=0.0,
        random_state=SEED,
    ),
    "NaiveBayes": NaiveBayes(alpha=0.5, n_bins=10),
}

svc_linear = SVC(C=1.0, kernel="linear", multiclass="ovo", gamma="scale", tol=1e-6)

probs_val = {}
for name, model in prob_models.items():
    model.fit(X_train_final, y_train)
    probs_val[name] = model.predict_proba(X_val_final)[:, 1]
    print(f"[Trained] {name}")

svc_linear.fit(X_train_final, y_train)
svc_scores_val = svc_linear.decision_function(X_val_final)
print("[Trained] SVC Linear")


def eval_cost_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    total_cost = int(C_FP * fp + C_FN * fn)
    avg_cost = total_cost / len(y_true)
    return {
        "FP": int(fp),
        "FN": int(fn),
        "Total Cost": total_cost,
        "Avg Cost": round(avg_cost, 4),
        "Precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_true, y_pred, zero_division=0), 4),
        "Macro F1": round(f1_score(y_true, y_pred, average="macro"), 4),
    }


rows = []
thresholds_grid = np.linspace(0.05, 0.95, 91)
threshold_curves = {}
empirical_opt = {}

for name, p1 in probs_val.items():
    # 1. Baseline (tau = 0.50)
    rows.append({"Model": name, "Strategi": "Baseline (τ=0.50)", **eval_cost_metrics(y_val, (p1 >= 0.50).astype(int))})
    # 2. Bayes Optimal (tau = 2/3)
    rows.append({"Model": name, "Strategi": f"Bayes Optimal (τ={TAU_BAYES:.3f})", **eval_cost_metrics(y_val, (p1 >= TAU_BAYES).astype(int))})
    # 3. Empirical Cost-Optimal
    costs = []
    for t in thresholds_grid:
        m = eval_cost_metrics(y_val, (p1 >= t).astype(int))
        costs.append(m["Avg Cost"])
    best_idx = int(np.argmin(costs))
    best_t = float(thresholds_grid[best_idx])
    threshold_curves[name] = np.array(costs)
    empirical_opt[name] = (best_t, float(costs[best_idx]))
    rows.append({"Model": name, "Strategi": f"Empirical Optimal (τ={best_t:.2f})", **eval_cost_metrics(y_val, (p1 >= best_t).astype(int))})

# SVC Linear (menggunakan margin score z)
rows.append({"Model": "SVC Linear", "Strategi": "Baseline (z>=0.00)", **eval_cost_metrics(y_val, (svc_scores_val >= 0.0).astype(int))})
z_grid = np.linspace(-0.5, 1.5, 201)
z_costs = [eval_cost_metrics(y_val, (svc_scores_val >= z).astype(int))["Total Cost"] for z in z_grid]
best_z = float(z_grid[int(np.argmin(z_costs))])
rows.append({"Model": "SVC Linear", "Strategi": f"Empirical Margin (z>={best_z:.2f})", **eval_cost_metrics(y_val, (svc_scores_val >= best_z).astype(int))})

cost_df = pd.DataFrame(rows)
display(cost_df)


In [ ]:
# Visualisasi 1: Perbandingan Total Cost & False Positive (Overbooking) sebelum vs sesudah
model_order = ["KNN", "DecisionTree", "LogisticRegression", "NaiveBayes", "SVC Linear"]
base_cost, opt_cost, base_fp, opt_fp = [], [], [], []

for m in model_order:
    sub = cost_df[cost_df["Model"] == m]
    base_row = sub.iloc[0]
    opt_row = sub.iloc[-1]
    base_cost.append(base_row["Total Cost"])
    opt_cost.append(opt_row["Total Cost"])
    base_fp.append(base_row["FP"])
    opt_fp.append(opt_row["FP"])

style_cfg = {
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans", "sans-serif"],
    "axes.linewidth": 2.0,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.frameon": False,
}

with plt.rc_context(style_cfg):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.4))
    fig.patch.set_facecolor("white")
    x = np.arange(len(model_order))
    w = 0.36

    # Panel Kiri: Total Cost
    b1 = ax1.bar(x - w / 2, base_cost, width=w, color=PALETTE["red_2"], edgecolor="black", linewidth=1.5, hatch="//", label="Baseline Symmetric")
    b2 = ax1.bar(x + w / 2, opt_cost, width=w, color=PALETTE["blue_main"], edgecolor="black", linewidth=1.5, label="Cost-Sensitive Optimal")
    for bar in b1:
        ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 25, f"{int(bar.get_height())}", ha="center", va="bottom", fontsize=10, fontweight="bold")
    for bar in b2:
        ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 25, f"{int(bar.get_height())}", ha="center", va="bottom", fontsize=10, fontweight="bold", color=PALETTE["blue_main"])
    ax1.set_xticks(x)
    ax1.set_xticklabels(model_order, rotation=15, ha="right")
    ax1.set_ylabel("Total Cost (2·FP + FN)")
    ax1.set_title("Total Cost Validation (C_FP=2, C_FN=1)", fontweight="bold", pad=10)
    ax1.set_ylim(0, max(base_cost) * 1.16)
    ax1.legend(loc="upper left", fontsize=10.5)

    # Panel Kanan: False Positive (Overbooking)
    f1 = ax2.bar(x - w / 2, base_fp, width=w, color=PALETTE["red_strong"], edgecolor="black", linewidth=1.5, hatch="//", label="Baseline FP (Overbooking)")
    f2 = ax2.bar(x + w / 2, opt_fp, width=w, color=PALETTE["green_3"], edgecolor="black", linewidth=1.5, label="Cost-Sensitive FP")
    for bar in f1:
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10, f"{int(bar.get_height())}", ha="center", va="bottom", fontsize=10, fontweight="bold")
    for bar in f2:
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10, f"{int(bar.get_height())}", ha="center", va="bottom", fontsize=10, fontweight="bold", color="#1E5631")
    ax2.set_xticks(x)
    ax2.set_xticklabels(model_order, rotation=15, ha="right")
    ax2.set_ylabel("Jumlah False Positive (Overbooking)")
    ax2.set_title("Penurunan Kasus Overbooking (False Positive)", fontweight="bold", pad=10)
    ax2.set_ylim(0, max(base_fp) * 1.16)
    ax2.legend(loc="upper left", fontsize=10.5)

    fig.tight_layout(pad=1.5)
    comp_path = OUTPUT_DIR / "cost_sensitive_comparison.png"
    fig.savefig(comp_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)

print(f"Grafik perbandingan cost-sensitive disimpan ke: {comp_path}")
display(Image(filename=str(comp_path)))


In [ ]:
# Visualisasi 2: Kurva Average Cost terhadap Decision Threshold (tau* = 2/3)
curve_colors = {
    "KNN": PALETTE["blue_main"],
    "DecisionTree": PALETTE["teal"],
    "LogisticRegression": PALETTE["violet"],
    "NaiveBayes": PALETTE["red_strong"],
}

with plt.rc_context(style_cfg):
    fig, ax = plt.subplots(figsize=(10.5, 5.4))
    fig.patch.set_facecolor("white")

    for name, costs in threshold_curves.items():
        c = curve_colors[name]
        best_t, best_c = empirical_opt[name]
        ax.plot(thresholds_grid, costs, lw=2.4, color=c, label=f"{name} (opt τ={best_t:.2f}, avg={best_c:.4f})")
        ax.scatter([best_t], [best_c], color=c, edgecolor="black", s=80, zorder=5)

    ax.axvline(0.50, color="#6B7280", linestyle=":", lw=1.8, label="Baseline Threshold (τ = 0.50)")
    ax.axvline(TAU_BAYES, color="#111827", linestyle="--", lw=2.0, label=r"Bayes Optimal ($\tau^* = 2/3 \approx 0.667$)")

    ax.set_xlabel(r"Decision Threshold ($\tau$)")
    ax.set_ylabel("Average Cost per Booking")
    ax.set_title(r"Kurva Average Cost terhadap Ambang Keputusan ($C_{FP}=2, C_{FN}=1$)", fontweight="bold", pad=10)
    ax.set_xlim(0.05, 0.95)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=2, fontsize=10.5)
    fig.tight_layout(pad=1.2)

    curve_path = OUTPUT_DIR / "cost_sensitive_threshold_curve.png"
    fig.savefig(curve_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)

print(f"Grafik kurva threshold disimpan ke: {curve_path}")
display(Image(filename=str(curve_path)))


### Ringkasan Hasil Bonus #7 (*Cost-Sensitive Classification*)
- Pergeseran ambang keputusan ke *Bayes Optimal* ($\tau^* = 2/3$) maupun *Empirical Cost-Optimal* secara konsisten menurunkan **Total Cost** dan memangkas **False Positive (Overbooking)** pada seluruh lima model.
- Model **KNN** *tuned* menghasilkan biaya terendah di sepanjang rentang ambang keputusan sekaligus memiliki kalibrasi probabilitas yang sangat baik (ambang empiris optimal berdekatan dengan $\tau^* = 2/3$).
- Model **Naive Bayes** membutuhkan ambang empiris sedikit lebih tinggi daripada $2/3$ karena sifat *overconfidence* akibat asumsi independensi kondisional antarfitur.
